# MT Benchmark — Turkish → English (MBART)

Evaluates `facebook/mbart-large-50-many-to-many-mmt` on Turkish→English translation.

**Dataset:** MaCoCu Turkish–English (Option A) or Tatoeba (Option B, no Drive required)  
**Metrics:** METEOR, BERTScore, XLMrScore  
**Results:** See [`README.md`](README.md)

This notebook is part of the `multilingual-mt` evaluation toolkit. For methodology see [`WORKFLOW.md`](../../WORKFLOW.md).

In [ ]:
# Uncomment when running on Colab
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score nltk accelerate datasets
import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, AutoTokenizer
from bert_score import score as bert_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
SAMPLE_SIZE = 10_000
BATCH_SIZE  = 16      # reduce if OOM
MAX_LENGTH  = 512
SRC_LANG    = 'tr_TR' # MBART language code for Turkish
TGT_LANG    = 'en_XX' # MBART language code for English
# ──────────────────────────────────────────────────────────────────────────────

## 1. Load dataset

**Option A — Tatoeba (public, no Drive required):** smaller, general-domain, immediate download.  
**Option B — MaCoCu (from Drive):** larger, web-crawled Turkish corpus. This is what the reference results in `README.md` are based on.

In [ ]:
# ── Option A: Tatoeba (no Drive) ─────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset('Helsinki-NLP/tatoeba_mt', 'tur-eng', split='test', trust_remote_code=True)
df = pd.DataFrame({
    'source': [r['sourceString'] for r in raw],
    'target': [r['targetString'] for r in raw],
})
df = df.dropna().sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)
print(f'{len(df):,} sentence pairs')
df.head(3)

In [ ]:
# ── Option B: MaCoCu from Drive (uncomment to use) ────────────────────────────
# FILE_PATH = '/content/drive/MyDrive/YOUR_PROJECT/data/MaCoCu-tr-en.sent.txt'
#
# raw = pd.read_csv(FILE_PATH, sep='\t', on_bad_lines='skip')
# raw = raw[raw['translation_direction'] == 'first-orig-second-ht']
# raw['bleualign_score']    = raw['bleualign_score'].astype(float)
# raw['bicleaner_ai_score'] = raw['bicleaner_ai_score'].astype(float)
# raw = raw[raw['bicleaner_ai_score'] > 0.9]
# raw = raw[raw['bleualign_score'] > 0.4]
# df = raw[['src_text', 'trg_text']].rename(columns={'src_text': 'source', 'trg_text': 'target'})
# df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=1).reset_index(drop=True)
# print(f'{len(df):,} sentence pairs')

In [ ]:
# Sentence length distribution
df['src_len'] = df['source'].str.len()
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df['src_len'], bins=60, color='#1B1A18', alpha=0.8)
ax.set_xlabel('Source sentence length (characters)')
ax.set_title('Sentence length distribution')
plt.tight_layout()
plt.show()
print(df['src_len'].describe().round(1))

## 2. Translate — MBART

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.data[idx], return_tensors='pt',
            padding='max_length', truncation=True, max_length=self.max_length,
        )
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze()}

In [ ]:
model_name = 'facebook/mbart-large-50-many-to-many-mmt'
tokenizer  = MBart50TokenizerFast.from_pretrained(model_name)
model      = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer.src_lang = SRC_LANG

model.to(DEVICE).eval()
print(f'Model loaded on {DEVICE}')

In [ ]:
dataset    = TranslationDataset(list(df['source']), tokenizer, MAX_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

translations = []
for batch in tqdm(dataloader):
    data = {k: v.to(DEVICE) for k, v in batch.items()}
    with torch.no_grad():
        tokens = model.generate(**data, forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG])
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))

df['translation'] = translations
print('Done.')

## 3. METEOR

In [ ]:
meteor_fn = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(df['target'], df['translation']), total=len(df)):
    scores.append(meteor_fn([ref.split()], hyp.split()))
df['meteor'] = scores
print(f'Average METEOR: {np.mean(scores):.4f}')

## 4. BERTScore

In [ ]:
tok  = AutoTokenizer.from_pretrained('roberta-large')
refs = [' '.join(tok.tokenize(r)) for r in df['target']]
cands = [' '.join(tok.tokenize(h)) for h in df['translation']]

_, _, F1 = bert_score(cands, refs, lang='en', model_type='roberta-large', verbose=True)
df['bertscore'] = F1.numpy()
print(f'Average BERTScore F1: {F1.mean():.4f}')

## 5. XLMrScore

Cross-lingual: compares the translation against the **source** Turkish text using `xlm-roberta-base`. A high score means the translation preserves the meaning of the original, independent of the gold standard.

In [ ]:
xlm_tok   = AutoTokenizer.from_pretrained('xlm-roberta-base')
src_tok   = [' '.join(xlm_tok.tokenize(s)) for s in df['source']]
hyp_tok   = [' '.join(xlm_tok.tokenize(h)) for h in df['translation']]

_, _, F1_xlm = bert_score(hyp_tok, src_tok, model_type='xlm-roberta-base', verbose=True)
df['xlmrscore'] = F1_xlm.numpy()
print(f'Average XLMrScore F1: {F1_xlm.mean():.4f}')

## 6. Results

In [ ]:
summary = df[['meteor', 'bertscore', 'xlmrscore']].describe().round(4)
print(summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes, ['meteor', 'bertscore', 'xlmrscore'], ['METEOR', 'BERTScore', 'XLMrScore']):
    ax.hist(df[col], bins=40, color='#1B1A18', alpha=0.8)
    ax.axvline(df[col].mean(), color='#C8A876', linestyle='--', label=f'mean={df[col].mean():.3f}')
    ax.set_title(label)
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Error analysis

In [ ]:
# Low BERTScore sample
low = df[df['bertscore'] < 0.9].sample(n=min(20, len(df[df['bertscore'] < 0.9])), random_state=1)

for _, row in low.iterrows():
    print(f'SOURCE:      {row["source"]}')
    print(f'GOLD:        {row["target"]}')
    print(f'TRANSLATION: {row["translation"]}')
    print(f'METEOR={row["meteor"]:.3f}  BERTScore={row["bertscore"]:.3f}  XLMrScore={row["xlmrscore"]:.3f}')
    print()

In [ ]:
# Save results (uncomment if Drive is mounted)
# df.to_csv('/content/drive/MyDrive/YOUR_PROJECT/outputs/MaCoCu_outputs-BART.csv', index=False)

print('Reference results from original experiment (10K MaCoCu, GPU A100, batch=16):')
print('  METEOR:    0.534')
print('  BERTScore: 0.929')
print('  XLMrScore: 0.852')
print('  GPU time:  41 min')